# 🕺 MimicMotion SOTA - Backend Realista (Abril 2026)
Este notebook utiliza a arquitetura **Confidence-aware Video Generation** para criar vídeos ultra-realistas no Colab T4.

In [ ]:
# @title 1. Instalação de Dependências de Ponta
import os
import sys
import shutil

%cd /content
if os.path.exists('MimicMotion'):
    shutil.rmtree('MimicMotion')

!git clone https://github.com/Tencent/MimicMotion.git
%cd /content/MimicMotion

# Instalação manual das versões compatíveis com o código atual da Tencent
!pip install -q fastapi uvicorn python-multipart xformers==0.0.25
!pip install -q diffusers==0.27.2 transformers accelerate decord einops omegaconf onnxruntime-gpu
!pip install -q moviepy opencv-python-headless

# Download dos pesos oficiais
from huggingface_hub import snapshot_download
os.makedirs('models', exist_ok=True)
snapshot_download(repo_id="Tencent/MimicMotion", local_dir="models", allow_patterns=["*.pth", "*.bin", "*.json"])

print("✅ Dependências e Modelos prontos!")

In [ ]:
%%writefile main.py
import os
import sys
import torch
import uuid
import shutil
import cv2
import numpy as np
import uvicorn
from fastapi import FastAPI, UploadFile, File
from fastapi.responses import FileResponse

# --- FIX DE IMPORTAÇÃO CONFORME REPOSITÓRIO TENCENT ---
# O repositório usa 'mimicmotion' (sem underline) como pacote principal
sys.path.append(os.getcwd())

try:
    # Caminho exato do GitHub: mimicmotion/pipelines/pipeline_mimicmotion.py
    from mimicmotion.pipelines.pipeline_mimicmotion import MimicMotionPipeline
except ImportError as e:
    print(f"Erro de importação: {e}. Tentando mapeamento alternativo...")
    sys.path.append(os.path.join(os.getcwd(), "mimicmotion"))
    from pipelines.pipeline_mimicmotion import MimicMotionPipeline

app = FastAPI()

# Configuração de Performance para GPU T4 (16GB VRAM)
pipe = MimicMotionPipeline.from_pretrained("models", torch_dtype=torch.float16)
pipe.to("cuda")
pipe.enable_model_cpu_offload() 
pipe.enable_vae_slicing()

def save_video(frames, output_path, fps=15):
    height, width, _ = frames[0].shape
    out = cv2.VideoWriter(output_path, cv2.VideoWriter_fourcc(*'mp4v'), fps, (width, height))
    for frame in frames:
        out.write(cv2.cvtColor(frame, cv2.COLOR_RGB2BGR))
    out.release()

@app.post("/generate")
async def generate(character_image: UploadFile = File(...), reference_video: UploadFile = File(...)):
    session_id = str(uuid.uuid4())
    work_dir = f"temp/{session_id}"
    os.makedirs(work_dir, exist_ok=True)
    
    img_path, vid_path, out_path = f"{work_dir}/i.png", f"{work_dir}/v.mp4", f"{work_dir}/o.mp4"
    with open(img_path, "wb") as f: shutil.copyfileobj(character_image.file, f)
    with open(vid_path, "wb") as f: shutil.copyfileobj(reference_video.file, f)

    # Geração com Realismo SOTA (Estilo Kling)
    result = pipe(
        ref_image=img_path,
        motion_video=vid_path,
        num_inference_steps=25, 
        guidance_scale=4.5,
        fps=15,
        chunk_size=8 # Crucial para não travar a T4
    )
    
    save_video(result.frames[0], out_path)
    return FileResponse(out_path, media_type="video/mp4")

if __name__ == "__main__":
    uvicorn.run(app, host="0.0.0.0", port=8000)

In [ ]:
# @title 3. Execução e Túnel
print("IP para senha do Localtunnel:")
!curl ipv4.icanhazip.com

!npm install -g localtunnel

import threading
import time
def run_lt():
    !lt --port 8000
threading.Thread(target=run_lt, daemon=True).start()

time.sleep(5)
!python main.py